# Results Consolidator

Merges per-model chunk files (`test_0_300.json`, `test_300_600.json`, `test_600_894.json`)  
into a single `test.json` per model × shot level.

Covers all 13 models × 4 shot levels (k = 0, 1, 2, 3).

**Debug files** (`test_0_1.json` etc.) are excluded — they contain a single record  
used during development and would duplicate the first row if merged.

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

In [ ]:
PROJECT_ROOT = Path("../").resolve()

SHOT_LEVELS         = [0, 1, 2, 3]
SUBSETS             = ["test"]   # add "dev" or "train" if those chunks exist
MIN_CHUNK_RECORDS   = 10         # files with fewer records are treated as debug runs
EXPECTED_RECORDS    = {"test": 894}

print(f"Project root : {PROJECT_ROOT}")
print(f"Shot levels  : {SHOT_LEVELS}")
print(f"Subsets      : {SUBSETS}")

In [ ]:
def _parse_chunk_range(filename: str, subset: str) -> Optional[Tuple[int, int]]:
    """Extract (start, end) from a chunk filename like 'test_0_300.json'."""
    m = re.fullmatch(rf"{re.escape(subset)}_(\d+)_(\d+)\.json", filename)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def get_chunk_files(results_path: Path, subset: str) -> List[Path]:
    """Return valid chunk files sorted by start index, excluding debug files."""
    candidates = []
    for f in results_path.glob(f"{subset}_*.json"):
        rng = _parse_chunk_range(f.name, subset)
        if rng is None:
            continue
        start, end = rng
        if (end - start) < MIN_CHUNK_RECORDS:
            continue          # skip debug single-record files like test_0_1.json
        candidates.append((start, f))
    candidates.sort(key=lambda x: x[0])
    return [f for _, f in candidates]


def merge_chunks(
    model_dir: Path,
    subset: str,
    expected_total: Optional[int] = None,
) -> Dict[str, Any]:
    """Merge all valid chunk files into {subset}.json. Returns a status dict."""
    out_path   = model_dir / f"{subset}.json"
    model_name = model_dir.name
    chunks     = get_chunk_files(model_dir, subset)

    if not chunks:
        return dict(model=model_name, status="no chunks found", records=0, chunks=[])

    merged: List[Dict] = []
    for cf in chunks:
        with open(cf) as fh:
            merged.extend(json.load(fh))

    with open(out_path, "w") as fh:
        json.dump(merged, fh, indent=4)

    warnings = []
    if expected_total is not None and len(merged) != expected_total:
        warnings.append(f"expected {expected_total}, got {len(merged)}")

    return dict(
        model   = model_name,
        status  = "ok" if not warnings else f"WARN: {'; '.join(warnings)}",
        records = len(merged),
        chunks  = [c.name for c in chunks],
    )

In [ ]:
all_results = []

for k in SHOT_LEVELS:
    results_dir = PROJECT_ROOT / f"results-{k}shot"
    if not results_dir.exists():
        print(f"[k={k}] Not found: {results_dir} — skipping")
        continue

    model_dirs = sorted(d for d in results_dir.iterdir() if d.is_dir())
    print(f"\nk={k}  ({len(model_dirs)} models)")

    for model_dir in model_dirs:
        for subset in SUBSETS:
            result = merge_chunks(
                model_dir     = model_dir,
                subset        = subset,
                expected_total= EXPECTED_RECORDS.get(subset),
            )
            result["k"]      = k
            result["subset"] = subset
            all_results.append(result)

            icon = "✓" if result["status"] == "ok" else "✗"
            print(f"  {icon}  {result['model']:<52}  {result['records']:>4} records  {result['status']}")

print("\nDone.")

In [ ]:
df = pd.DataFrame(all_results)

# Pivot: rows = model, columns = k
pivot = (
    df[df["subset"] == "test"]
    .pivot(index="model", columns="k", values="records")
    .rename(columns=lambda k: f"k={k}")
)
pivot.index.name = "model"

print(f"Record counts — test set (expected {EXPECTED_RECORDS['test']} per cell):\n")
print(pivot.to_string())

anomalies = df[
    (df["subset"] == "test") &
    (df["records"] != EXPECTED_RECORDS["test"])
]
if anomalies.empty:
    print(f"\nAll cells: {EXPECTED_RECORDS['test']} records. Ready for eval pipeline.")
else:
    print("\nANOMALIES:")
    print(anomalies[["k", "model", "records", "status"]].to_string(index=False))

In [ ]:
# Verify required fields are present in a sample merged file
REQUIRED_FIELDS = [
    "context", "question", "is_impossible", "answer_text", "question_id",
    "answer_extracted", "answerable", "reasoning",
    "input_tokens", "output_tokens", "total_tokens",
]

sample_model = sorted(
    d for d in (PROJECT_ROOT / "results-0shot").iterdir() if d.is_dir()
)[0]
sample_file = sample_model / "test.json"

with open(sample_file) as fh:
    sample_data = json.load(fh)

actual  = set(sample_data[0].keys())
missing = set(REQUIRED_FIELDS) - actual
extra   = actual - set(REQUIRED_FIELDS)

print(f"Sample : {sample_file.relative_to(PROJECT_ROOT)}  ({len(sample_data)} records)")
print(f"Fields : {sorted(actual)}")
print(f"Missing: {missing or 'none'}")
print(f"Extra  : {extra or 'none'}")